# R6 YOLO12l Cat Detector â€” Tuned + Trained from CSV

- Model: YOLO12l (26M params; ~50MB on disk)
- Hyperparameter tuning: Ultralytics `model.tune()` â€” genetic algorithm over 18 params, 50 iterations Ã— 20 epochs
- Final training: 300 epochs with the best mutation, cosine LR, patience=50
- Eval: standard mAP + recall sweep across confidence thresholds (the bot's threshold is 0.552)

Deploy target: Modal T4 (~30ms/inference vs YOLO12s's ~10ms â€” irrelevant next to encoder pass).
Train on A100. T4 can train at batch=4 but takes ~12h+ â€” not recommended.

In [ ]:
!pip -q install "ultralytics>=8.3.200" pandas tqdm pillow

In [ ]:
import os, json, math, random, zipfile, shutil, time
from collections import defaultdict
from pathlib import Path
import pandas as pd
import torch
from tqdm.notebook import tqdm
from google.colab import drive
drive.mount("/content/drive")

SEED = 42
random.seed(SEED)

DRIVE_PATH = "/content/drive/MyDrive"
RUN_DIR = os.path.join(DRIVE_PATH, "R6_yolo12l")
os.makedirs(RUN_DIR, exist_ok=True)

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
print(f"GPU: {props.name}, VRAM: {vram_gb:.1f} GB")

# Batch tuning for YOLO12l at imgsz=640
if vram_gb >= 70:   BATCH = 64
elif vram_gb >= 35: BATCH = 32
elif vram_gb >= 14: BATCH = 8
else:               BATCH = 4
print(f"BATCH={BATCH}")

In [ ]:
ZIP_NAME = "R6_TomCat_Training.zip"
ZIP_PATH = os.path.join(DRIVE_PATH, ZIP_NAME)
assert os.path.exists(ZIP_PATH), f"Zip not found at: {ZIP_PATH}"
print(f"Found: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB)")

RAW_DIR = "/content/raw_data"
if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)
print("Unzipping...")
with zipfile.ZipFile(ZIP_PATH, "r") as zf: zf.extractall(RAW_DIR)

csv_path = img_folder = None
for root, dirs, files in os.walk(RAW_DIR):
    for fn in files:
        if fn.lower().endswith('.csv') and 'pic' in fn.lower():
            csv_path = os.path.join(root, fn)
    for d in dirs:
        if 'totalpics' in d.lower():
            img_folder = os.path.join(root, d)
assert csv_path and img_folder, f"Missing CSV or TotalPicsOfCats"
print(f"CSV: {csv_path}\nImages: {img_folder}")

ok_ext = {".jpg", ".jpeg", ".png"}
sn_to_path = {Path(fn).stem.lower(): os.path.join(img_folder, fn) for fn in os.listdir(img_folder) if Path(fn).suffix.lower() in ok_ext}
print(f"Indexed {len(sn_to_path)} images")

In [ ]:
# Build YOLO-format dataset from CSV.
# All cats collapse to one class (class_id=0); bot does re-ID separately.
YDS = "/content/R6_yolo_dataset"
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    p = os.path.join(YDS, sub)
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p, exist_ok=True)

# Group boxes per image: a single sn may appear in multiple CSV rows.
img_boxes = defaultdict(list)
skip_norow = skip_rej = skip_bad = 0
df = pd.read_csv(csv_path)
for _, row in df.iterrows():
    sn = str(row['Serial Number']).strip().lower()
    box_coords = str(row.get('Box Coordinates', '')).strip()
    if box_coords in ('Rejected','','nan'):
        skip_rej += 1; continue
    if sn not in sn_to_path:
        skip_norow += 1; continue
    for coord in box_coords.split('|'):
        parts = coord.strip().split()
        if len(parts) != 4: skip_bad += 1; continue
        try: cx, cy, bw, bh = map(float, parts)
        except: skip_bad += 1; continue
        if not (0 < bw <= 1 and 0 < bh <= 1 and 0 < cx < 1 and 0 < cy < 1):
            skip_bad += 1; continue
        img_boxes[sn].append((cx, cy, bw, bh))

print(f"Images with boxes: {len(img_boxes)}")
print(f"Total boxes: {sum(len(v) for v in img_boxes.values())}")
print(f"Skipped: rejected={skip_rej} no_image={skip_norow} bad_box={skip_bad}")

# Random 85/15 split by image (not by box).
sns = sorted(img_boxes.keys())
random.seed(SEED); random.shuffle(sns)
n_val = int(0.15 * len(sns))
val_sns = set(sns[:n_val]); train_sns = set(sns[n_val:])
print(f"Split: train={len(train_sns)} val={len(val_sns)}")

# Copy images (symlinks would be faster, but Ultralytics is finicky about them).
for sn in tqdm(sns, desc="Writing dataset"):
    split = "val" if sn in val_sns else "train"
    img_src = sn_to_path[sn]; ext = Path(img_src).suffix.lower()
    img_dst = os.path.join(YDS, f"images/{split}", f"{sn}{ext}")
    shutil.copy2(img_src, img_dst)
    label_dst = os.path.join(YDS, f"labels/{split}", f"{sn}.txt")
    with open(label_dst, "w") as f:
        for (cx, cy, bw, bh) in img_boxes[sn]:
            f.write(f"0 {cx} {cy} {bw} {bh}\n")

data_yaml = os.path.join(YDS, "data.yaml")
with open(data_yaml, "w") as f:
    f.write(f"path: {YDS}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['cat']\n")
print(f"data.yaml: {data_yaml}")

In [ ]:
# Sanity: count files and peek at one label.
for split in ["train", "val"]:
    n_img = len(os.listdir(os.path.join(YDS, f"images/{split}")))
    n_lbl = len(os.listdir(os.path.join(YDS, f"labels/{split}")))
    print(f"{split}: {n_img} images, {n_lbl} labels")

sample = sorted(os.listdir(os.path.join(YDS, "labels/train")))[0]
print(f"\nSample label ({sample}):")
print(open(os.path.join(YDS, "labels/train", sample)).read())

## Smoke test â€” one direct training call

`model.tune()` runs each iteration in a subprocess and only surfaces `returned non-zero exit status 1` on failure â€” the real error is hidden. This cell does a single `model.train()` call so any training-time exception (OOM, corrupt image, bad label, dependency mismatch) prints with a full traceback. **Run this first.** If it completes one epoch successfully, the tune call below will too.

In [ ]:
from ultralytics import YOLO
import shutil, os

SMOKE_DIR = '/content/runs/smoke'
if os.path.exists(SMOKE_DIR): shutil.rmtree(SMOKE_DIR)

smoke_model = YOLO('yolo12l.pt')
smoke_model.train(
    data=data_yaml,
    epochs=1,
    imgsz=640,
    batch=BATCH,
    cache='disk',   # writes preprocessed cache to /content; safer than 'ram'
    workers=2,
    plots=False,
    val=True,
    project='/content/runs', name='smoke', exist_ok=True,
)
print('\nâœ… smoke test passed â€” safe to run tune below.')

## Tune â€” 50 GA iterations Ã— 20 epochs each

Ultralytics' `.tune()` mutates 18 hyperparams (LR, momentum, weight decay, warmup, loss gains, and all augmentation strengths) via a genetic algorithm. Each iteration is a 20-epoch training run; the best mutation seeds the next.

Expected wall time on A100 40GB: ~3-4 hours.

In [ ]:
from ultralytics import YOLO

# Keep tune working dir on local SSD â€” Drive sync chokes on per-iteration writes.
LOCAL_RUNS = "/content/runs"
if os.path.exists(LOCAL_RUNS): shutil.rmtree(LOCAL_RUNS)
os.makedirs(LOCAL_RUNS, exist_ok=True)

model = YOLO('yolo12l.pt')
model.tune(
    data=data_yaml,
    epochs=8,
    iterations=10,
    optimizer='AdamW',
    imgsz=640,
    batch=BATCH,
    cache='disk',    # preprocessed cache on /content SSD; safer than ram across subprocesses
    workers=4,      # Colab CPU is limited; default 8 can stall
    plots=False, save=False, val=True,
    project=LOCAL_RUNS, name='tune',
)

# Copy run artifacts (incl. best_hyperparameters.yaml) to Drive.
TUNE_DST = os.path.join(RUN_DIR, "tune")
if os.path.exists(TUNE_DST): shutil.rmtree(TUNE_DST)
shutil.copytree(os.path.join(LOCAL_RUNS, "tune"), TUNE_DST)
best_hyp = os.path.join(TUNE_DST, "best_hyperparameters.yaml")
print(f"\nBest hyperparams: {best_hyp}")
print(open(best_hyp).read())


## Final train â€” 300 epochs with the best mutation

Cosine LR + early stopping at patience=50. Checkpoint every 50 epochs in case of disconnect.

In [ ]:
model = YOLO('yolo12l.pt')
model.train(
    data=data_yaml,
    cfg=best_hyp,
    epochs=100,
    imgsz=640,
    batch=BATCH,
    cache='disk',
    workers=4,
    patience=25,
    cos_lr=True,
    save_period=25,
    project=LOCAL_RUNS, name='final',
)

# Copy run + best.pt to Drive.
FINAL_DST = os.path.join(RUN_DIR, "final")
if os.path.exists(FINAL_DST): shutil.rmtree(FINAL_DST)
shutil.copytree(os.path.join(LOCAL_RUNS, "final"), FINAL_DST)
best_pt = os.path.join(FINAL_DST, "weights", "best.pt")
print(f"Final best weights: {best_pt}")


## Eval â€” recall sweep at varied confidence thresholds

Standard mAP@0.5 + recall at the bot's actual threshold (`cv.detect_conf=0.552`).

In [ ]:
model = YOLO(best_pt)
print("Default eval:")
metrics = model.val(data=data_yaml, conf=0.001, plots=False, verbose=False)
print(f"  mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")

print("\nRecall sweep (bot uses conf=0.552):")
for c in [0.3, 0.4, 0.5, 0.552, 0.6, 0.7]:
    m = model.val(data=data_yaml, conf=c, plots=False, verbose=False)
    print(f"  conf={c:.3f}: P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f}")

In [ ]:
# Copy the production weight to a stable name in Drive.
final_path = os.path.join(RUN_DIR, "R6_cat_yolo12l.pt")
shutil.copy2(best_pt, final_path)
print(f"Saved: {final_path}")
print(f"Size: {os.path.getsize(final_path)/1e6:.1f} MB")